In [8]:
"""

"""

# imports
import os, sys

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
boxdir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/'
datadir = boxdir + 'data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

proj_crs = 'EPSG:5070'

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation


In [9]:
# --- Load the incidents (perimeters)
incis = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg')
incis = gpd.read_file(incis)

# --- Keep only forested incidents
incis = incis.copy()
incis = incis[
    (incis['FUEL_MODEL'].str.contains('Timber', case=False)) |
    (incis['tcc']>=10) # keep forested incidents
]

print(f"{len(incis)} incidents in the subset (timber fires).")

4259 incidents in the subset (timber fires).


In [10]:
incis.columns

Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT_NUM_SITREPS',
       'INJURIES_TOTAL', 'IRWIN_ID', 'LL_CONFIDENCE', 'LL_UPDATE',
       'LOCAL_TIMEZONE', 'MTBS_FIRE_NAME', 'MTBS_ID',
       'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'PEAK_EVACUATIONS',
       'POO_CITY', 'POO_COUNTY', 'POO_LATITUDE', 'POO_LONGITUDE',
       'POO_SHORT_LOCATION_DESC', 'POO_S

In [11]:
# --- Drop some columns
incis_ = incis.copy()
incis_.drop(columns=[
    'system:index','START_YEAR_y','.geo','IRWIN_C','FOD_C','inci_id'
], inplace=True)
incis_.rename(columns={'START_YEAR_x':'START_YEAR'}, inplace=True)
incis_.columns

Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT_NUM_SITREPS',
       'INJURIES_TOTAL', 'IRWIN_ID', 'LL_CONFIDENCE', 'LL_UPDATE',
       'LOCAL_TIMEZONE', 'MTBS_FIRE_NAME', 'MTBS_ID',
       'NWCG_CAUSE_CLASSIFICATION', 'NWCG_GENERAL_CAUSE', 'PEAK_EVACUATIONS',
       'POO_CITY', 'POO_COUNTY', 'POO_LATITUDE', 'POO_LONGITUDE',
       'POO_SHORT_LOCATION_DESC', 'POO_S

In [13]:
# --- Load the NSI summary
fp = os.path.join(projdir,"data/tabular/NSI_summary.csv")
nsi_summary = pd.read_csv(fp)
print(nsi_summary.columns)

# --- Merge to the incidents table
incis_m = pd.merge(incis_, nsi_summary, on='INCIDENT_ID', how='left')
print(incis_m.columns)

Index(['INCIDENT_ID', 'val_struct_NR', 'val_struct_R', 'n_struct_NR',
       'n_struct_R', 'n_struct_total', 'prop_R', 'val_struct_NR24',
       'val_struct_R24'],
      dtype='object')
Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       'FINAL_REPORT_DATE', 'FIRE_SIZE_CLASS', 'FIRE_YEAR', 'FOD_CONT_DATE',
       'FOD_CONT_DOY', 'FOD_CONT_TIME', 'FOD_DISCOVERY_DATE',
       'FOD_DISCOVERY_DOY', 'FOD_DISCOVERY_TIME', 'FOD_EXCL_CAT',
       'FOD_FIRE_SIZE', 'FOD_ID', 'FOD_INCIDENT_GROUP_NAME', 'FOD_LATITUDE',
       'FOD_LONGITUDE', 'FUEL_MODEL', 'INCIDENT_DESCRIPTION', 'INCIDENT_ID',
       'INCIDENT_ID_OLD', 'INCIDENT_NAME', 'INCIDENT_NUMBER',
       'INCTYP_ABBREVIATION', 'INC_IDENTIFIER', 'INC_MGMT_NUM_SITREPS',
       'INJURIES_TOTAL', 'IRWIN_ID', 'LL_CONFIDENCE', 'LL_UPDATE',
       'LOCAL_TIMEZONE', 'MTBS_FIRE_NAME', 'MTBS_ID'

In [22]:
# --- Load the TWIG summary
fp = os.path.join(projdir, 'data/tabular/TWIG_summary_wide-format.csv')
twig_summary = pd.read_csv(fp)
print(twig_summary.columns)

# --- Merge
incis_m_ = pd.merge(incis_m, twig_summary, on='INCIDENT_ID', how='left')
incis_m_.columns

Index(['INCIDENT_ID', 'Biomass Removal', 'Broadcast Burn', 'Fire Use',
       'Hand Pile', 'Hand Pile Burn', 'Jackpot Burn', 'Lop and Scatter',
       'Machine Pile', 'Machine Pile Burn', 'Manual Thin', 'Mastication',
       'Mechanical Thin', 'Thin + Rx Fire', 'Wildfire'],
      dtype='object')


Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       ...
       'Hand Pile Burn', 'Jackpot Burn', 'Lop and Scatter', 'Machine Pile',
       'Machine Pile Burn', 'Manual Thin', 'Mastication', 'Mechanical Thin',
       'Thin + Rx Fire', 'Wildfire'],
      dtype='object', length=112)

In [25]:
# Wildfire Risk to Communities
fp = os.path.join(projdir,'data/tabular/WRC_hurisk_popden_weighted.csv')
wrc_hurisk_popden = pd.read_csv(fp)
wrc_hurisk_popden.drop(columns='Unnamed: 0', inplace=True)
print(wrc_hurisk_popden.columns)

# --- Merge to the table
incis_m_f = pd.merge(incis_m_, wrc_hurisk_popden, on='INCIDENT_ID', how='left')
print(incis_m_f.columns)

Index(['INCIDENT_ID', 'WRC_HURisk_wt', 'WRC_PopDen_wt'], dtype='object')
Index(['CAUSE', 'COMPLEX', 'DISCOVERY_DATE', 'DISCOVERY_DOY',
       'EVACUATION_REPORTED', 'EXPECTED_CONTAINMENT_DATE', 'FATALITIES',
       'FATALITIES_PUBLIC', 'FATALITIES_RESPONDER', 'FINAL_ACRES',
       ...
       'Lop and Scatter', 'Machine Pile', 'Machine Pile Burn', 'Manual Thin',
       'Mastication', 'Mechanical Thin', 'Thin + Rx Fire', 'Wildfire',
       'WRC_HURisk_wt', 'WRC_PopDen_wt'],
      dtype='object', length=114)


In [26]:
print(len(incis_m_f))

4259


In [27]:
# --- Save this file out
out_fp = os.path.join(projdir,'data/tabular/ics209plus_2014to2023_summaries_draft022026.csv')
incis_m_f.to_csv(out_fp, index=False)
print(f"Saved to {out_fp}")

# spatial data
out_fp = os.path.join(projdir,'data/spatial/mod/ics209plus_2014to2023_summaries_draft022026.gpkg')
incis_m_f.to_file(out_fp, index=False)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/ics209plus_2014to2023_summaries_draft022026.csv
Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/spatial/mod/ics209plus_2014to2023_summaries_draft022026.gpkg


In [29]:
for c in incis_m_f.columns:
    print(c)

CAUSE
COMPLEX
DISCOVERY_DATE
DISCOVERY_DOY
EVACUATION_REPORTED
EXPECTED_CONTAINMENT_DATE
FATALITIES
FATALITIES_PUBLIC
FATALITIES_RESPONDER
FINAL_ACRES
FINAL_REPORT_DATE
FIRE_SIZE_CLASS
FIRE_YEAR
FOD_CONT_DATE
FOD_CONT_DOY
FOD_CONT_TIME
FOD_DISCOVERY_DATE
FOD_DISCOVERY_DOY
FOD_DISCOVERY_TIME
FOD_EXCL_CAT
FOD_FIRE_SIZE
FOD_ID
FOD_INCIDENT_GROUP_NAME
FOD_LATITUDE
FOD_LONGITUDE
FUEL_MODEL
INCIDENT_DESCRIPTION
INCIDENT_ID
INCIDENT_ID_OLD
INCIDENT_NAME
INCIDENT_NUMBER
INCTYP_ABBREVIATION
INC_IDENTIFIER
INC_MGMT_NUM_SITREPS
INJURIES_TOTAL
IRWIN_ID
LL_CONFIDENCE
LL_UPDATE
LOCAL_TIMEZONE
MTBS_FIRE_NAME
MTBS_ID
NWCG_CAUSE_CLASSIFICATION
NWCG_GENERAL_CAUSE
PEAK_EVACUATIONS
POO_CITY
POO_COUNTY
POO_LATITUDE
POO_LONGITUDE
POO_SHORT_LOCATION_DESC
POO_STATE
PROJECTED_FINAL_IM_COST
START_YEAR
STR_DAMAGED_COMM_TOTAL
STR_DAMAGED_RES_TOTAL
STR_DAMAGED_TOTAL
STR_DESTROYED_COMM_TOTAL
STR_DESTROYED_RES_TOTAL
STR_DESTROYED_TOTAL
STR_THREATENED_COMM_MAX
STR_THREATENED_MAX
STR_THREATENED_RES_MAX
SUPPRESSION_MET